# Kaggle Runner — Tubes Besar 2 IF3270
Jalankan notebook ini dari atas ke bawah. Semua output tersimpan ke file repo dan di-commit ke GitHub di akhir.

**Prasyarat Kaggle:**
- Accelerator: **GPU T4** (Settings → Accelerator)
- Dataset attached:
  - `puneet6060/intel-image-classification` (untuk FASE 1 — CNN)
  - `adityajn105/flickr8k` (untuk FASE 2 — Captioning)
- Secret: `GITHUB_TOKEN` (Add-ons → Secrets)

---
## FASE 0 — Setup

In [ ]:
import os

REPO = '/kaggle/working/repo'
!git clone https://github.com/wrdtlkhoir/Tubes2ML-17-k01 {REPO}
%cd {REPO}

!pip install -q papermill rouge-score

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

import sys
sys.path.insert(0, REPO)

os.makedirs('weights/cnn',     exist_ok=True)
os.makedirs('weights/rnn_lstm', exist_ok=True)
os.makedirs('results/plots',   exist_ok=True)
os.makedirs('results/tables',  exist_ok=True)
os.makedirs('data/features',   exist_ok=True)
os.makedirs('data/captions',   exist_ok=True)

print('Setup selesai')

---
## FASE 1 — Person A: CNN (Intel Image Classification)

Dataset: 25k gambar, 6 kelas (buildings, forest, glacier, mountain, sea, street).  
Output: `weights/cnn/*.h5`, macro F1-score tiap variasi, loss plots.

> Skip ke FASE 2 kalau `weights/cnn/` sudah terisi.

In [ ]:
# Path dataset Intel Image Classification di Kaggle
INTEL_BASE  = '/kaggle/input/intel-image-classification'
INTEL_TRAIN = f'{INTEL_BASE}/seg_train/seg_train'
INTEL_VAL   = f'{INTEL_BASE}/seg_test/seg_test'   # pakai seg_test sebagai val
INTEL_TEST  = f'{INTEL_BASE}/seg_test/seg_test'

# Cek struktur
import os
classes = sorted(os.listdir(INTEL_TRAIN))
print(f'Kelas ({len(classes)}): {classes}')
for c in classes:
    n = len(os.listdir(f'{INTEL_TRAIN}/{c}'))
    print(f'  {c}: {n} gambar')

In [ ]:
# Training semua variasi Conv2D + LocallyConnected2D
# Menggunakan src/cnn/train.py:ModelTrainer
#
# Patch kecil: wrap train_model() supaya history tersimpan untuk plotting
import tensorflow as tf
from src.cnn.train import ModelTrainer

cnn_histories = {}   # {model_name: history.history}
_orig_train = ModelTrainer.train_model

def _patched_train(self, model, train_ds, val_ds, model_name, epochs=20):
    history = _orig_train(self, model, train_ds, val_ds, model_name, epochs)
    cnn_histories[model_name] = history.history
    return history

ModelTrainer.train_model = _patched_train

trainer_cnn = ModelTrainer(
    train_dir  = INTEL_TRAIN,
    val_dir    = INTEL_VAL,
    test_dir   = INTEL_TEST,
    output_dir = 'weights/cnn',
)

train_ds, val_ds, test_ds = trainer_cnn.load_data(img_size=(224, 224), batch_size=32)
print(f'Kelas: {trainer_cnn.class_names}')

# ~1-2 jam di T4 untuk semua variasi
trainer_cnn.run_all_training(train_ds, val_ds, test_ds)
print('CNN training selesai')

In [ ]:
# Plot training/val loss semua variasi CNN
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

def plot_loss_group(histories_subset: dict, title: str, save_path: str):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, h in histories_subset.items():
        label = name.replace('conv2d_', '').replace('locally_connected2d_', 'lc_')
        axes[0].plot(h['loss'],     label=label)
        axes[1].plot(h['val_loss'], label=label)
    for ax, ylabel in zip(axes, ['Train Loss', 'Val Loss']):
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# Pisah history berdasarkan variasi
layer_hist   = {k: v for k, v in cnn_histories.items() if 'layers_' in k}
filter_hist  = {k: v for k, v in cnn_histories.items() if 'filters_' in k}
kernel_hist  = {k: v for k, v in cnn_histories.items() if 'kernels_' in k}
lc_hist      = {k: v for k, v in cnn_histories.items() if 'locally' in k}

plot_loss_group(layer_hist,  'CNN Loss — Variasi Jumlah Layer',       'results/plots/cnn_loss_num_layers.png')
plot_loss_group(filter_hist, 'CNN Loss — Variasi Jumlah Filter',      'results/plots/cnn_loss_num_filters.png')
plot_loss_group(kernel_hist, 'CNN Loss — Variasi Ukuran Filter',      'results/plots/cnn_loss_filter_size.png')

# Pooling: pisah dari layer_hist berdasarkan nama
pool_hist = {k: v for k, v in cnn_histories.items() if 'pool_' in k}
plot_loss_group(pool_hist,   'CNN Loss — Variasi Jenis Pooling',      'results/plots/cnn_loss_pooling_type.png')

# Shared vs non-shared
all_hist = {**layer_hist, **lc_hist}
plot_loss_group(all_hist,    'CNN Loss — Shared (Conv2D) vs Non-shared (LC2D)', 'results/plots/cnn_shared_vs_nonshared.png')

print('Loss plots CNN tersimpan di results/plots/')

In [ ]:
# Simpan semua F1 scores ke tabel
import json

with open('weights/cnn/training_results.json') as f:
    cnn_results = json.load(f)

import pandas as pd
rows = []
for variant_type, variants in cnn_results.items():
    for name, info in variants.items():
        rows.append({'model': name, 'macro_f1': info['macro_f1']})

df_cnn = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
display(df_cnn)

best_cnn_name = df_cnn.iloc[0]['model']
print(f'\nArsitektur CNN terbaik: {best_cnn_name}')

In [ ]:
# A4 — Simpan histories + evaluasi from scratch (Keras vs Scratch, Shared vs Non-Shared)
import json, os, pandas as pd
from src.cnn.evaluate import ModelEvaluator

os.makedirs('results/tables', exist_ok=True)

# 1. Simpan loss histories ke disk (dibutuhkan cnn_eval.ipynb standalone)
with open('results/tables/cnn_training_histories.json', 'w') as f:
    json.dump(cnn_histories, f, indent=2)
print(f'Histories saved: {list(cnn_histories.keys())}')

# 2. Evaluasi from scratch: best Conv2D (shared) vs LocallyConnected2D (non-shared)
evaluator_a4 = ModelEvaluator('weights/cnn')
eval_rows = []
for model_name in [best_cnn_name, 'locally_connected2d_baseline']:
    path = f'weights/cnn/{model_name}.h5'
    if not os.path.exists(path):
        print(f'Skip {model_name} — tidak ditemukan'); continue
    res = evaluator_a4.evaluate_model_pair(model_name, test_ds)
    row = {
        'model':       model_name,
        'type':        'Conv2D (shared)' if 'locally' not in model_name else 'LC2D (non-shared)',
        'keras_f1':    round(res['keras_metrics']['macro_f1'],        4),
        'scratch_f1':  round(res['from_scratch_metrics']['macro_f1'], 4),
        'params':      res['parameter_count'],
        'preds_match': res['predictions_match'],
    }
    eval_rows.append(row)
    print(f"{row['type']}: Keras F1={row['keras_f1']}  Scratch F1={row['scratch_f1']}  "
          f"Params={row['params']:,}  Match={row['preds_match']}")

with open('results/tables/cnn_shared_vs_nonshared.json', 'w') as f:
    json.dump(eval_rows, f, indent=2)
display(pd.DataFrame(eval_rows))
print('A4 selesai')

---
## FASE 2 — Person B: Feature Extraction + Preprocessing + Training Decoder

CNN encoder untuk captioning: **InceptionV3** frozen (bukan dari weights Person A).  
Person A's CNN hanya untuk Intel Image Classification.

> Skip ke FASE 3 kalau `weights/rnn_lstm/` dan `data/features/` sudah terisi.

In [ ]:
# FASE 2 Step 1 — Feature Extraction (InceptionV3, frozen ImageNet)
# Menggunakan src/cnn/utils.py:extract_features()
from src.cnn.utils import extract_features
import glob

IMAGE_DIR = '/kaggle/input/flickr8k/Images'

image_paths = glob.glob(f'{IMAGE_DIR}/*.jpg')
print(f'{len(image_paths)} gambar Flickr8k ditemukan')

# ~20-30 menit di T4 GPU
extract_features(
    image_paths,
    output_path = 'data/features',
    model_name  = 'InceptionV3',
    target_size = (299, 299),
    batch_size  = 64,
)
print('Feature extraction selesai')

In [ ]:
# FASE 2 Step 2 — Preprocessing Caption & Buat Splits
import pandas as pd, json, random
random.seed(42)

CAPTIONS_FILE = '/kaggle/input/flickr8k/captions.txt'

df = pd.read_csv(CAPTIONS_FILE)
print('Columns:', df.columns.tolist())
print(df.head(2))

In [ ]:
# Sesuaikan nama kolom jika berbeda
IMG_COL = 'image'
CAP_COL = 'caption'

all_names = df[IMG_COL].unique().tolist()
random.shuffle(all_names)
n = len(all_names)
splits = {
    'train': all_names[:int(0.75 * n)],
    'val':   all_names[int(0.75 * n):int(0.875 * n)],
    'test':  all_names[int(0.875 * n):],
}
print({k: len(v) for k, v in splits.items()})

for split, names in splits.items():
    caps = {name: df[df[IMG_COL] == name][CAP_COL].tolist() for name in names}
    with open(f'data/captions/{split}_captions.json', 'w') as f:
        json.dump(caps, f)

print('Caption splits tersimpan')

In [ ]:
# Gabungkan individual feature .npy → per-split dict (dibutuhkan experiment notebooks)
import numpy as np, os

for split_name, names in splits.items():
    feat_dict = {}
    for name in names:
        npy_p = os.path.join('data/features',
                             name.replace('.jpg', '.npy').replace('.png', '.npy'))
        if os.path.exists(npy_p):
            feat_dict[name] = np.load(npy_p)
    out_p = f'data/features/{split_name}_features.npy'
    np.save(out_p, feat_dict)
    print(f'{split_name}: {len(feat_dict)} features → {out_p}')

print('Feature dict files siap untuk experiment notebooks')

In [ ]:
# FASE 2 Step 4 — Training 12 Variasi Decoder (6 RNN + 6 LSTM)
# Menggunakan src/rnn/train.py:CaptioningTrainer
from src.rnn.train import CaptioningTrainer

rnn_histories = {}  # untuk plot loss
from src.rnn.train import CaptioningTrainer
_orig_rnn_train = CaptioningTrainer._train_one

def _patched_rnn_train(self, model, model_name, train_ds, val_ds):
    result = _orig_rnn_train(self, model, model_name, train_ds, val_ds)
    rnn_histories[model_name] = result['history']
    return result

CaptioningTrainer._train_one = _patched_rnn_train

trainer_rnn = CaptioningTrainer(
    features_dir     = 'data/features',
    captions_file    = CAPTIONS_FILE,
    train_split      = splits['train'],
    val_split        = splits['val'],
    test_split       = splits['test'],
    output_dir       = 'weights/rnn_lstm',
    vocab_path       = 'data/captions/vocab.json',
    embed_dim        = 256,
    max_caption_len  = 30,
    epochs           = 20,
    batch_size       = 64,
    patience         = 5,
)

# ~3-5 jam di T4
trainer_rnn.run(injection='pre', rnn_types=('rnn', 'lstm'))
print('Training decoder selesai')

In [ ]:
# Plot training/val loss semua variasi RNN & LSTM
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

def plot_rnn_loss_group(histories_subset: dict, title: str, save_path: str):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, h in histories_subset.items():
        label = name.split('_preinject_')[1] if '_preinject_' in name else name
        axes[0].plot(h['loss'],     label=label)
        axes[1].plot(h['val_loss'], label=label)
    for ax, ylabel in zip(axes, ['Train Loss', 'Val Loss']):
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

rnn_h  = {k: v for k, v in rnn_histories.items() if k.startswith('rnn_')}
lstm_h = {k: v for k, v in rnn_histories.items() if k.startswith('lstm_')}

# Pisah berdasarkan num_layers dan hidden_size
rnn_layer_h  = {k: v for k, v in rnn_h.items()  if '_H128' in k or '_H512' in k}
lstm_layer_h = {k: v for k, v in lstm_h.items() if '_H128' in k or '_H512' in k}

plot_rnn_loss_group(rnn_layer_h,  'RNN Loss — Variasi Layer & Hidden Size',  'results/plots/rnn_loss_num_layers.png')
plot_rnn_loss_group(lstm_layer_h, 'LSTM Loss — Variasi Layer & Hidden Size', 'results/plots/lstm_loss_num_layers.png')

# Hidden size comparison
rnn_h128 = {k: v for k, v in rnn_h.items()  if '_H128' in k}
rnn_h512 = {k: v for k, v in rnn_h.items()  if '_H512' in k}
plot_rnn_loss_group({**rnn_h128, **rnn_h512},  'RNN Loss — Variasi Hidden Size',  'results/plots/rnn_loss_hidden_size.png')

lstm_h128 = {k: v for k, v in lstm_h.items() if '_H128' in k}
lstm_h512 = {k: v for k, v in lstm_h.items() if '_H512' in k}
plot_rnn_loss_group({**lstm_h128, **lstm_h512}, 'LSTM Loss — Variasi Hidden Size', 'results/plots/lstm_loss_hidden_size.png')

print('Loss plots RNN/LSTM tersimpan di results/plots/')

---
## FASE 3 — Person C: Eksperimen & Evaluasi

Menjalankan 4 notebook yang sudah dibuat menggunakan **papermill**.  
Output (gambar, tabel, cell outputs) tersimpan langsung ke `.ipynb` untuk di-commit.

In [ ]:
# Verifikasi semua input tersedia sebelum eksperimen
import glob, os

weights  = glob.glob('weights/rnn_lstm/*.h5')
vocab_ok = os.path.exists('data/captions/vocab.json')
feat_combined = all(
    os.path.exists(f'data/features/{s}_features.npy')
    for s in ('train', 'val', 'test')
)

print(f'Decoder weights     : {len(weights)} file')
print(f'vocab.json          : {vocab_ok}')
print(f'Combined features   : {feat_combined}  (train/val/test_features.npy)')

assert len(weights) >= 12, 'Kurang dari 12 weight files — cek training FASE 2'
assert vocab_ok,           'vocab.json tidak ada — cek training FASE 2'
assert feat_combined,      'Combined feature files belum ada — cek sel feature combination FASE 2'

In [ ]:
# FASE 3 Step 2 — Run semua 12 variasi
import papermill as pm

pm.execute_notebook(
    'src/experiments/run_all_variants.ipynb',
    'src/experiments/run_all_variants.ipynb',
)
print('run_all_variants selesai')

In [ ]:
# FASE 3 Step 3 — Keras vs Scratch
# Baca BLEU-4 terbaik dan patch ke notebook sebelum dijalankan
import json, nbformat

with open('results/tables/all_variants_scores.json') as f:
    scores = json.load(f)

rnn_best  = max((k for k in scores if k.startswith('rnn_')),  key=lambda k: scores[k]['bleu4'])
lstm_best = max((k for k in scores if k.startswith('lstm_')), key=lambda k: scores[k]['bleu4'])

def variant_to_weight(name):
    # 'rnn_2layer_512' → 'weights/rnn_lstm/rnn_preinject_L2_H512.h5'
    parts  = name.split('_')            # ['rnn', '2layer', '512']
    rtype  = parts[0]                   # rnn / lstm
    layers = parts[1][0]                # '2'
    hidden = parts[2]                   # '512'
    return f'weights/rnn_lstm/{rtype}_preinject_L{layers}_H{hidden}.h5'

best_rnn_path  = variant_to_weight(rnn_best)
best_lstm_path = variant_to_weight(lstm_best)
print(f'Best RNN:  {rnn_best}  →  {best_rnn_path}')
print(f'Best LSTM: {lstm_best}  →  {best_lstm_path}')

def patch_notebook(nb_path, replacements):
    with open(nb_path) as f:
        nb = nbformat.read(f, as_version=4)
    for cell in nb.cells:
        if cell.cell_type == 'code':
            for old, new in replacements.items():
                cell.source = cell.source.replace(old, new)
    with open(nb_path, 'w') as f:
        nbformat.write(nb, f)

patch_notebook('src/experiments/compare_keras_scratch.ipynb', {
    "'../../weights/rnn_lstm/rnn_preinject_L?_H???.h5'":  f"'../../{best_rnn_path}'",
    "'../../weights/rnn_lstm/lstm_preinject_L?_H???.h5'": f"'../../{best_lstm_path}'",
    "'rnn_Xlayer_YYY'":  f"'{rnn_best}'",
    "'lstm_Xlayer_YYY'": f"'{lstm_best}'",
})

pm.execute_notebook(
    'src/experiments/compare_keras_scratch.ipynb',
    'src/experiments/compare_keras_scratch.ipynb',
)
print('compare_keras_scratch selesai')

In [ ]:
# FASE 3 Step 4 — Caption length ablation
with open('results/tables/keras_vs_scratch.json') as f:
    cmp = json.load(f)

best_model   = max(cmp, key=lambda r: r['bleu4'])
ablation_type = 'lstm' if 'lstm' in best_model['model'].lower() else 'rnn'
ablation_w    = best_lstm_path if ablation_type == 'lstm' else best_rnn_path
print(f"Ablation model: {best_model['model']}  ({ablation_type})")

patch_notebook('src/experiments/caption_length_ablation.ipynb', {
    "'../../weights/rnn_lstm/lstm_preinject_L?_H???.h5'": f"'../../{ablation_w}'",
    "BEST_TYPE    = 'lstm'": f"BEST_TYPE    = '{ablation_type}'",
})

pm.execute_notebook(
    'src/experiments/caption_length_ablation.ipynb',
    'src/experiments/caption_length_ablation.ipynb',
)
print('caption_length_ablation selesai')

In [ ]:
# FASE 3 Step 5 — Qualitative Analysis
patch_notebook('src/experiments/qualitative_analysis.ipynb', {
    "'../../weights/rnn_lstm/rnn_preinject_L?_H???.h5'":   f"'../../{best_rnn_path}'",
    "'../../weights/rnn_lstm/lstm_preinject_L?_H???.h5'": f"'../../{best_lstm_path}'",
    "'../../data/images/test/'": f"'{IMAGE_DIR}/",
})

pm.execute_notebook(
    'src/experiments/qualitative_analysis.ipynb',
    'src/experiments/qualitative_analysis.ipynb',
)
print('qualitative_analysis selesai')

In [ ]:
# FASE 3 Step 5 — Unit Tests: CNN From Scratch (A1)
pm.execute_notebook(
    'src/cnn/cnn_scratch.ipynb',
    'src/cnn/cnn_scratch.ipynb',
)
print('cnn_scratch unit tests selesai')

In [ ]:
# FASE 3 Step 6 — Unit Tests: RNN/LSTM From Scratch (B3)
pm.execute_notebook(
    'src/rnn/rnn_lstm_scratch.ipynb',
    'src/rnn/rnn_lstm_scratch.ipynb',
)
print('rnn_lstm_scratch unit tests selesai')

In [ ]:
# FASE 3 Step 7 — CNN Evaluation Notebook (A4)
# Patch INTEL_TEST_DIR ke path Kaggle sebelum dijalankan
patch_notebook('src/cnn/cnn_eval.ipynb', {
    "'../../data/archive-6/seg_test/seg_test'": f"'{INTEL_TEST}'",
})

pm.execute_notebook(
    'src/cnn/cnn_eval.ipynb',
    'src/cnn/cnn_eval.ipynb',
)
print('cnn_eval selesai')

---
## FASE 4 — Push Semua Hasil ke GitHub

In [ ]:
import os

token = os.environ.get('GITHUB_TOKEN', '')
if not token:
    raise EnvironmentError('Set GITHUB_TOKEN di Kaggle Secrets (Add-ons → Secrets)')

!git config user.email "sbimasena@gmail.com"
!git config user.name "SAKTI BIMASENA"
!git remote set-url origin https://{token}@github.com/wrdtlkhoir/Tubes2ML-17-k01.git

# Notebooks dengan output cells + results + captions
!git add src/experiments/*.ipynb
!git add src/cnn/cnn_scratch.ipynb src/cnn/cnn_eval.ipynb
!git add src/rnn/rnn_lstm_scratch.ipynb
!git add results/
!git add data/captions/

!git status
!git commit -m "feat: experiment results — CNN & RNN/LSTM all variants, unit tests, A4 eval"
!git push origin main
print('Push ke GitHub selesai!')

---
## Ringkasan Output

| File | Isi |
|---|---|
| `weights/cnn/*.h5` | CNN models Intel Image Classification (tidak di-commit) |
| `weights/rnn_lstm/*.h5` | 12 decoder models (tidak di-commit) |
| `results/plots/cnn_loss_*.png` | Loss plots semua variasi CNN |
| `results/plots/rnn_loss_*.png` + `lstm_loss_*.png` | Loss plots semua variasi decoder |
| `results/plots/bleu4_*.png` | BLEU-4 comparison bar charts |
| `results/plots/qualitative_analysis.png` | Grid ≥10 gambar dengan captions |
| `results/tables/all_variants_scores.json` | BLEU-4/METEOR semua 12 variasi |
| `results/tables/keras_vs_scratch.json` | Perbandingan 4 model |
| `results/tables/cnn_shared_vs_nonshared.json` | Perbandingan shared vs non-shared CNN |
| `src/experiments/*.ipynb` | Notebooks dengan output cells terisi |